In [0]:
from pyspark.sql.functions import *

GOLD_DB = "gold"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

print("Gold database ready.")

In [0]:
transactions = spark.table("silver.transactions_enriched")

In [0]:
customer_spending_summary = (
    transactions
        .groupBy("customer_id")
        .agg(
            count("*").alias("total_transactions"),
            sum("amount_usd").alias("total_spend_usd"),
            avg("amount_usd").alias("avg_transaction_usd"),
            min("transaction_timestamp").alias("first_transaction"),
            max("transaction_timestamp").alias("last_transaction")
        )
)

In [0]:
customers = spark.table("bronze.customers")

customer_spending_summary = (
    customer_spending_summary.alias("spend")
    .join(
        customers.alias("cust"),
        col("spend.customer_id") == col("cust.customer_id"),
        "left"
    )
    .select(
        col("spend.customer_id"),
        col("cust.first_name"),
        col("cust.last_name"),
        col("cust.country"),
        col("total_transactions"),
        col("total_spend_usd"),
        col("avg_transaction_usd"),
        col("first_transaction"),
        col("last_transaction")
    )
)

In [0]:
(
    customer_spending_summary.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("gold.customer_spending_summary")
)

print("Gold table created: gold.customer_spending_summary")